# Enhanced Graph RAG System

## Overview
This notebook implements an enhanced Graph RAG methodology with the following improvements:

1. **Enhanced Intent Classifier**: Classifies both intents AND artifacts (not just intents)
2. **Graph Formation**: Stores only replies (not full email context) with unique email IDs
3. **New Graph RAG Search**: Finds emails in intersection of intents and artifacts (at least 1 intent AND 1 artifact)
4. **Removed Style**: No writing style collection/retrieval (simplified approach)
5. **FAQ Search**: Kept as-is for content retrieval

## Key Changes from Previous Version:
- Intent classifier now returns both intents and artifacts
- Graph edges store only reply text (not full email context)
- Graph search uses intersection logic (intents ∩ artifacts)
- No style-based retrieval


In [2]:
# Imports and Setup
import json
import pandas as pd
import networkx as nx
import uuid
from typing import List, Dict, Set, Tuple
from itertools import combinations, product

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import ollama

print("✅ All imports loaded successfully!")


/Users/zubair/Desktop/Dev/ai-automation-agent/email-agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports loaded successfully!


In [3]:
# Load FAQ data
faq = pd.read_csv("../data/faq_updated.csv", encoding="utf-8")

# Load email pairs with labels
with open("../data/generated_email_pairs.json", "r", encoding="utf-8") as f:
    labels = json.load(f)

print(f"✅ FAQ rows: {len(faq)}")
print(f"✅ Labelled email pairs: {len(labels)}")
print(f"✅ Sample item keys: {list(labels[0].keys()) if labels else 'No data'}")

# Display sample item structure
if labels:
    print("\n📋 Sample Item Structure:")
    sample = labels[0]
    print(f"  ID: {sample.get('id')}")
    print(f"  Subject: {sample.get('subject')}")
    print(f"  Labels: {sample.get('labels')}")
    print(f"  Has reply: {'reply' in sample and sample['reply'] is not None}")


✅ FAQ rows: 22
✅ Labelled email pairs: 104
✅ Sample item keys: ['id', 'labels', 'subject', 'sender_email', 'prospect_email', 'reply']

📋 Sample Item Structure:
  ID: eef2f02b-5d07-45dd-9098-b4fe094cfd29
  Subject: Meeting to Discuss Research Collaboration
  Labels: {'topic': 'Professor/Academic', 'intents': ['schedule'], 'artifacts': ['calendly']}
  Has reply: True


In [4]:
# Extract all unique intents and artifacts from data
unique_intents = set()
unique_artifacts = set()

for item in labels:
    label_data = item.get("labels", {})
    for intent in label_data.get("intents", []):
        unique_intents.add(intent)
    for artifact in label_data.get("artifacts", []):
        unique_artifacts.add(artifact)

print("📊 Unique Intents:")
for intent in sorted(unique_intents):
    print(f"  - {intent}")

print(f"\n📊 Unique Artifacts ({len(unique_artifacts)} total):")
for artifact in sorted(unique_artifacts):
    print(f"  - {artifact}")


📊 Unique Intents:
  - accept_or_decline
  - confirm
  - request_feedback
  - request_info
  - reschedule
  - schedule
  - send_materials
  - share_feedback

📊 Unique Artifacts (10 total):
  - calendly
  - draft
  - ds_resume
  - github
  - linkedin_profile
  - phone_number
  - portfolio
  - report
  - swe_resume
  - zoom_link


In [5]:
# ARTIFACT DICTIONARY
# This dictionary explains what each artifact means - used by the classifier

ARTIFACT_DICTIONARY = {
    "ds_resume": "Data Science resume - use when email mentions ML, data science, analytics, or data-related roles",
    "swe_resume": "Software Engineering resume - use when email mentions software engineering, development, or SWE roles",
    "linkedin_profile": "LinkedIn profile link - use when email asks for LinkedIn, professional profile, or networking",
    "portfolio": "Portfolio website - use when email asks for work samples, projects, or portfolio",
    "github": "GitHub profile - use when email asks for code samples, GitHub, or technical projects",
    "calendly": "Calendly scheduling link - use when email asks to schedule, book, or arrange a meeting",
    "zoom_link": "Zoom meeting link - use when email asks for video call, Zoom meeting, or virtual meeting",
    "phone_number": "Phone number - use when email asks for phone contact or urgent communication",
    "draft": "Draft document - use when email asks for draft, document review, or written materials",
    "report": "Report document - use when email asks for report, progress update, or formal document",
}

print("📚 Artifact Dictionary:")
print(f"Total artifacts defined: {len(ARTIFACT_DICTIONARY)}\n")
for artifact, description in sorted(ARTIFACT_DICTIONARY.items()):
    print(f"  {artifact}: {description}")


📚 Artifact Dictionary:
Total artifacts defined: 10

  calendly: Calendly scheduling link - use when email asks to schedule, book, or arrange a meeting
  draft: Draft document - use when email asks for draft, document review, or written materials
  ds_resume: Data Science resume - use when email mentions ML, data science, analytics, or data-related roles
  github: GitHub profile - use when email asks for code samples, GitHub, or technical projects
  linkedin_profile: LinkedIn profile link - use when email asks for LinkedIn, professional profile, or networking
  phone_number: Phone number - use when email asks for phone contact or urgent communication
  portfolio: Portfolio website - use when email asks for work samples, projects, or portfolio
  report: Report document - use when email asks for report, progress update, or formal document
  swe_resume: Software Engineering resume - use when email mentions software engineering, development, or SWE roles
  zoom_link: Zoom meeting link - use

In [6]:
# ENHANCED INTENT + ARTIFACT CLASSIFIER
# This classifier returns BOTH intents AND artifacts

def classify_intent_and_artifacts(
    email_text: str, 
    available_intents: List[str], 
    available_artifacts: List[str],
    artifact_dict: Dict[str, str]
) -> Dict[str, List[str]]:
    """
    Classify both intents AND artifacts from an email using LLM.
    
    Returns:
        {
            "intents": [list of intents],
            "artifacts": [list of artifacts]  # NEW: artifacts that will be useful
        }
    """
    intents_str = ", ".join(available_intents)
    artifacts_str = ", ".join(available_artifacts)
    
    # Build artifact descriptions for the prompt
    artifact_descriptions = "\n".join([
        f"- {artifact}: {desc}"
        for artifact, desc in sorted(artifact_dict.items())
        if artifact in available_artifacts
    ])
    
    prompt = f"""You are an email intent and artifact classifier.

Available intents (choose from these EXACT labels):
{intents_str}

Available artifacts (choose from these EXACT labels):
{artifacts_str}

Artifact meanings (use these to decide which artifacts are needed):
{artifact_descriptions}

Email to classify:
\"\"\"{email_text}\"\"\"

Instructions:
1. Identify ALL relevant intents from the intent list
2. Identify which artifacts will be useful for responding to this email
   - Be precise: if it asks for ML/analytics resume, return ONLY ds_resume
   - If it asks for software engineering resume, return ONLY swe_resume
   - Only return artifacts that are directly relevant
   - Keep the list concise and to the point
3. Return ONLY a JSON object with this exact format:
{{
    "intents": ["intent1", "intent2"],
    "artifacts": ["artifact1", "artifact2"]
}}

Examples:
- "Can you send me your ML resume?" → {{"intents": ["send_materials"], "artifacts": ["ds_resume"]}}
- "Share your resume and let's schedule a call" → {{"intents": ["send_materials", "schedule"], "artifacts": ["ds_resume", "calendly"]}}
- "What time works for a meeting?" → {{"intents": ["schedule"], "artifacts": ["calendly"]}}
- "Can I get your LinkedIn?" → {{"intents": ["request_info"], "artifacts": ["linkedin_profile"]}}

Return ONLY valid JSON, nothing else:
"""
    
    try:
        response = ollama.chat(
            model="qwen2.5:14b-instruct-q4_K_M",
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response["message"]["content"].strip()
        
        # Debug: print what LLM returned
        print(f"🔍 LLM raw response: {content[:300]}...")
        
        # Try to extract JSON from markdown code blocks
        import re
        
        # Remove markdown code blocks if present
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```\s*', '', content)
        content = content.strip()
        
        try:
            result = json.loads(content)
            if isinstance(result, dict):
                intents = result.get("intents", [])
                artifacts = result.get("artifacts", [])
                
                # Filter to only valid intents and artifacts
                valid_intents = [i for i in intents if i in available_intents]
                valid_artifacts = [a for a in artifacts if a in available_artifacts]
                
                print(f"✅ Classified intents: {valid_intents}")
                print(f"✅ Classified artifacts: {valid_artifacts}")
                
                return {
                    "intents": valid_intents if valid_intents else ["general_inquiry"],
                    "artifacts": valid_artifacts
                }
        except json.JSONDecodeError:
            # Try to extract JSON object from text
            match = re.search(r'\{.*?\}', content, re.DOTALL)
            if match:
                try:
                    result = json.loads(match.group())
                    if isinstance(result, dict):
                        intents = result.get("intents", [])
                        artifacts = result.get("artifacts", [])
                        valid_intents = [i for i in intents if i in available_intents]
                        valid_artifacts = [a for a in artifacts if a in available_artifacts]
                        print(f"✅ Classified intents: {valid_intents}")
                        print(f"✅ Classified artifacts: {valid_artifacts}")
                        return {
                            "intents": valid_intents if valid_intents else ["general_inquiry"],
                            "artifacts": valid_artifacts
                        }
                except:
                    pass
        
        # Fallback: try to find intent/artifact names in the response
        found_intents = []
        found_artifacts = []
        
        for intent in available_intents:
            if intent.lower() in content.lower():
                found_intents.append(intent)
        
        for artifact in available_artifacts:
            if artifact.lower() in content.lower():
                found_artifacts.append(artifact)
        
        if found_intents or found_artifacts:
            print(f"✅ Classified (fallback) intents: {found_intents}")
            print(f"✅ Classified (fallback) artifacts: {found_artifacts}")
            return {
                "intents": found_intents if found_intents else ["general_inquiry"],
                "artifacts": found_artifacts
            }
        
        print("⚠️ Could not parse intents/artifacts, using defaults")
        return {
            "intents": ["general_inquiry"],
            "artifacts": []
        }
        
    except Exception as e:
        print(f"⚠️ Error classifying intent/artifacts: {e}")
        return {
            "intents": ["general_inquiry"],
            "artifacts": []
        }

print("✅ Enhanced classifier function loaded!")


✅ Enhanced classifier function loaded!


In [7]:
# TEST: Enhanced Intent + Artifact Classifier
# Test the classifier with various email examples

test_emails = [
    "Hi Zubair, I'm interested in your ML background. Can you share your resume?",
    "Can we schedule a meeting to discuss the project?",
    "What's your LinkedIn profile? I'd like to connect.",
    "I need your data science resume for a position at Google.",
    "Share your software engineering resume and let's set up a call.",
    "Can I get your GitHub and portfolio links?"
]

print("🧪 Testing Enhanced Intent + Artifact Classifier\n")
print("="*80)

for i, email in enumerate(test_emails, 1):
    print(f"\n📧 Test Email {i}:")
    print(f"   {email}\n")
    
    result = classify_intent_and_artifacts(
        email_text=email,
        available_intents=list(unique_intents),
        available_artifacts=list(unique_artifacts),
        artifact_dict=ARTIFACT_DICTIONARY
    )
    
    print(f"   ✅ Intents: {result['intents']}")
    print(f"   ✅ Artifacts: {result['artifacts']}")
    print("-"*80)


🧪 Testing Enhanced Intent + Artifact Classifier


📧 Test Email 1:
   Hi Zubair, I'm interested in your ML background. Can you share your resume?

🔍 LLM raw response: {
    "intents": ["send_materials"],
    "artifacts": ["ds_resume"]
}...
✅ Classified intents: ['send_materials']
✅ Classified artifacts: ['ds_resume']
   ✅ Intents: ['send_materials']
   ✅ Artifacts: ['ds_resume']
--------------------------------------------------------------------------------

📧 Test Email 2:
   Can we schedule a meeting to discuss the project?

🔍 LLM raw response: {
    "intents": ["schedule"],
    "artifacts": ["calendly"]
}...
✅ Classified intents: ['schedule']
✅ Classified artifacts: ['calendly']
   ✅ Intents: ['schedule']
   ✅ Artifacts: ['calendly']
--------------------------------------------------------------------------------

📧 Test Email 3:
   What's your LinkedIn profile? I'd like to connect.

🔍 LLM raw response: {
    "intents": ["request_info"],
    "artifacts": ["linkedin_profile"]
}...
✅ 

## Graph Formation

**Key Changes:**
1. Store ONLY the reply (not full email context)
2. Create unique email ID for each email (already exists in data)
3. Store email_id on edges for later retrieval
4. Build graph with topic → intent and topic → artifact relationships


In [8]:
# Build Graph with ONLY replies (not full email context)
# Store email_id on edges for retrieval
# Generate unique email_id if it doesn't exist in data

G = nx.DiGraph()

# Mapping: email_id -> reply (for quick lookup later)
email_id_to_reply = {}

# Mapping: node_name -> set of email_ids (for finding emails connected to nodes)
node_to_email_ids = {}  # {node_name: set([email_id1, email_id2, ...])}

# Track ID generation statistics
ids_generated = 0
ids_existing = 0

print("🔨 Building graph with replies only...\n")

for idx, item in enumerate(labels):
    label_data = item.get("labels", {})
    topic = label_data.get("topic")
    intents = label_data.get("intents", [])
    artifacts = label_data.get("artifacts", [])
    
    # Get ONLY the reply (not full email context)
    reply = item.get("reply", "")
    
    # Get or generate email_id
    # Check if "id" field exists and is not empty
    email_id = item.get("id")
    if not email_id or email_id == "":
        # Generate unique UUID if id doesn't exist or is empty
        email_id = str(uuid.uuid4())
        ids_generated += 1
        # Optionally update the item in memory (for debugging)
        item["id"] = email_id
    else:
        ids_existing += 1
    
    # Store email_id -> reply mapping (always store, even if reply is empty)
    email_id_to_reply[email_id] = reply
    
    if not topic and not intents and not artifacts:
        continue
    
    # Add topic node
    if topic:
        G.add_node(topic, type="topic")
        if topic not in node_to_email_ids:
            node_to_email_ids[topic] = set()
        node_to_email_ids[topic].add(email_id)
    
    # Add intents as nodes and edges (with email_id)
    for intent in intents:
        G.add_node(intent, type="intent")
        if topic:
            # Store email_id on edge
            G.add_edge(topic, intent, relation="HAS_INTENT", email_id=email_id)
        
        if intent not in node_to_email_ids:
            node_to_email_ids[intent] = set()
        node_to_email_ids[intent].add(email_id)
    
    # Add artifacts as nodes and edges (with email_id)
    for artifact in artifacts:
        G.add_node(artifact, type="artifact")
        if topic:
            # Store email_id on edge
            G.add_edge(topic, artifact, relation="USES_ARTIFACT", email_id=email_id)
        
        if artifact not in node_to_email_ids:
            node_to_email_ids[artifact] = set()
        node_to_email_ids[artifact].add(email_id)

print(f"✅ Graph built:")
print(f"   Nodes: {len(G.nodes())}")
print(f"   Edges: {len(G.edges())}")
print(f"   Email ID -> Reply mappings: {len(email_id_to_reply)}")
print(f"   Nodes with email associations: {len(node_to_email_ids)}")
print(f"\n📊 ID Generation Statistics:")
print(f"   Existing IDs: {ids_existing}")
print(f"   Generated IDs: {ids_generated}")
print(f"   Total emails processed: {len(labels)}")

# Display sample structure
print(f"\n📋 Sample node structure:")
if G.nodes():
    sample_node = list(G.nodes())[0]
    print(f"   Node: {sample_node}")
    print(f"   Type: {G.nodes[sample_node].get('type')}")
    print(f"   Associated emails: {len(node_to_email_ids.get(sample_node, set()))}")

# Display sample edge with email_id
print(f"\n📋 Sample edge structure:")
if G.edges():
    sample_edge = list(G.edges(data=True))[0]
    print(f"   Edge: {sample_edge[0]} -> {sample_edge[1]}")
    print(f"   Relation: {sample_edge[2].get('relation')}")
    print(f"   Email ID: {sample_edge[2].get('email_id')}")
    
    # Show sample reply for this email_id
    sample_email_id = sample_edge[2].get('email_id')
    if sample_email_id in email_id_to_reply:
        sample_reply = email_id_to_reply[sample_email_id]
        print(f"   Sample Reply (first 100 chars): {sample_reply[:100]}...")


🔨 Building graph with replies only...

✅ Graph built:
   Nodes: 23
   Edges: 38
   Email ID -> Reply mappings: 104
   Nodes with email associations: 23

📊 ID Generation Statistics:
   Existing IDs: 104
   Generated IDs: 0
   Total emails processed: 104

📋 Sample node structure:
   Node: Professor/Academic
   Type: topic
   Associated emails: 25

📋 Sample edge structure:
   Edge: Professor/Academic -> schedule
   Relation: HAS_INTENT
   Email ID: d5a96f3e-e60e-43eb-96f4-75b035910a19
   Sample Reply (first 100 chars): Dear Professor Rogers,

Thank you for offering to meet with me. Please use the link https://calendly...


## New Graph RAG Search Methodology

**Key Change:** Instead of looking at all successors/predecessors, we now:
1. Find emails that exist in the intersection of intents and artifacts
2. Use exhaustive P&C (permutations & combinations) unless very huge
3. Require at least 1 intent AND 1 artifact match
4. Extract replies from matching emails


In [9]:
# NEW GRAPH RAG SEARCH FUNCTION
# Finds emails in intersection of intents and artifacts

def find_emails_by_intent_artifact_intersection(
    intents: List[str],
    artifacts: List[str],
    node_to_email_ids: Dict[str, Set[str]],
    max_combinations: int = 100  # Limit if combinations get too large
) -> List[str]:
    """
    Find email IDs that exist in the intersection of intents and artifacts.
    
    Logic:
    - For each intent, get all associated email_ids
    - For each artifact, get all associated email_ids
    - Find emails that appear in BOTH intent sets AND artifact sets
    - Use exhaustive P&C unless very huge
    
    Args:
        intents: List of intent labels
        artifacts: List of artifact labels
        node_to_email_ids: Mapping from node names to sets of email_ids
        max_combinations: Maximum number of combinations to try (safety limit)
    
    Returns:
        List of email_ids that match the intersection criteria
    """
    if not intents or not artifacts:
        print("⚠️ Need at least 1 intent AND 1 artifact for intersection search")
        return []
    
    print(f"\n🔍 Searching for emails with:")
    print(f"   Intents: {intents}")
    print(f"   Artifacts: {artifacts}")
    
    # Get email_ids for each intent
    intent_email_sets = []
    for intent in intents:
        if intent in node_to_email_ids:
            email_set = node_to_email_ids[intent]
            intent_email_sets.append(email_set)
            print(f"   Intent '{intent}': {len(email_set)} emails")
        else:
            print(f"   ⚠️ Intent '{intent}' not found in graph")
    
    # Get email_ids for each artifact
    artifact_email_sets = []
    for artifact in artifacts:
        if artifact in node_to_email_ids:
            email_set = node_to_email_ids[artifact]
            artifact_email_sets.append(email_set)
            print(f"   Artifact '{artifact}': {len(email_set)} emails")
        else:
            print(f"   ⚠️ Artifact '{artifact}' not found in graph")
    
    if not intent_email_sets or not artifact_email_sets:
        print("   ⚠️ No matching emails found (missing intents or artifacts)")
        return []
    
    # Find intersection: emails that appear in at least one intent set AND at least one artifact set
    # This is the key logic: intersection of (intent1 OR intent2 OR ...) AND (artifact1 OR artifact2 OR ...)
    
    # Union of all intent email sets (emails matching ANY intent)
    intent_union = set()
    for email_set in intent_email_sets:
        intent_union.update(email_set)
    
    # Union of all artifact email sets (emails matching ANY artifact)
    artifact_union = set()
    for email_set in artifact_email_sets:
        artifact_union.update(email_set)
    
    # Intersection: emails that match at least 1 intent AND at least 1 artifact
    matching_emails = intent_union.intersection(artifact_union)
    
    print(f"\n   ✅ Found {len(matching_emails)} emails in intersection")
    
    # If we have many matches, we can also try more specific combinations
    # (emails matching specific intent+artifact pairs)
    if len(matching_emails) > 0 and len(intents) * len(artifacts) <= max_combinations:
        print(f"\n   🔍 Trying specific intent+artifact combinations...")
        specific_matches = set()
        
        for intent in intents:
            if intent not in node_to_email_ids:
                continue
            intent_emails = node_to_email_ids[intent]
            
            for artifact in artifacts:
                if artifact not in node_to_email_ids:
                    continue
                artifact_emails = node_to_email_ids[artifact]
                
                # Emails matching this specific intent+artifact pair
                pair_matches = intent_emails.intersection(artifact_emails)
                if pair_matches:
                    specific_matches.update(pair_matches)
                    print(f"      '{intent}' + '{artifact}': {len(pair_matches)} emails")
        
        if specific_matches:
            print(f"   ✅ Specific combinations: {len(specific_matches)} emails")
            # Use specific matches if available (more precise)
            return list(specific_matches)
    
    return list(matching_emails)

print("✅ Graph RAG search function loaded!")


✅ Graph RAG search function loaded!


In [10]:
# TEST: Graph RAG Search
# Test the new intersection search methodology

print("🧪 Testing Graph RAG Search (Intent + Artifact Intersection)\n")
print("="*80)

test_cases = [
    {
        "intents": ["send_materials"],
        "artifacts": ["ds_resume"],
        "description": "ML resume request"
    },
    {
        "intents": ["schedule"],
        "artifacts": ["calendly"],
        "description": "Schedule meeting"
    },
    {
        "intents": ["send_materials", "schedule"],
        "artifacts": ["ds_resume", "calendly"],
        "description": "Multiple intents and artifacts"
    },
    {
        "intents": ["request_info"],
        "artifacts": ["linkedin_profile"],
        "description": "LinkedIn request"
    }
]

for i, test_case in enumerate(test_cases, 1):
    print(f"\n📋 Test Case {i}: {test_case['description']}")
    print(f"   Intents: {test_case['intents']}")
    print(f"   Artifacts: {test_case['artifacts']}\n")
    
    matching_email_ids = find_emails_by_intent_artifact_intersection(
        intents=test_case["intents"],
        artifacts=test_case["artifacts"],
        node_to_email_ids=node_to_email_ids
    )
    
    print(f"\n   ✅ Found {len(matching_email_ids)} matching emails")
    
    # Show sample replies
    if matching_email_ids:
        print(f"\n   📧 Sample replies (first 2):")
        for email_id in matching_email_ids[:2]:
            reply = email_id_to_reply.get(email_id, "No reply found")
            print(f"      Email ID: {email_id[:20]}...")
            print(f"      Reply: {reply[:150]}...")
            print()
    
    print("-"*80)


🧪 Testing Graph RAG Search (Intent + Artifact Intersection)


📋 Test Case 1: ML resume request
   Intents: ['send_materials']
   Artifacts: ['ds_resume']


🔍 Searching for emails with:
   Intents: ['send_materials']
   Artifacts: ['ds_resume']
   Intent 'send_materials': 25 emails
   Artifact 'ds_resume': 10 emails

   ✅ Found 5 emails in intersection

   🔍 Trying specific intent+artifact combinations...
      'send_materials' + 'ds_resume': 5 emails
   ✅ Specific combinations: 5 emails

   ✅ Found 5 matching emails

   📧 Sample replies (first 2):
      Email ID: 4cf5eb70-1416-4980-b...
      Reply: Dear Jane,

Thank you for reaching out! I'm excited about the opportunity at TechCorp. Please find my Data Science Resume attached: https://drive.goog...

      Email ID: 708038e4-9ea4-4581-8...
      Reply: Hello Sarah,

Thank you for considering my application at BrightFuture. Please find attached my Data Science Resume: https://drive.google.com/file/d/1...

------------------------------

## FAQ Search Setup
Keep FAQ search as-is (no changes)


In [12]:
# Setup Qdrant and Embedder
# Handle lock file issues if another instance is running

import os
import time

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Try to initialize Qdrant client
# If locked, try to remove lock file (only if safe to do so)
qdrant_path = "qdrant_data"
lock_file_path = os.path.join(qdrant_path, ".lock")

qdrant = None
max_retries = 3

for attempt in range(max_retries):
    try:
        qdrant = QdrantClient(path=qdrant_path)
        print(f"✅ Qdrant client initialized successfully")
        break
    except RuntimeError as e:
        if "already accessed by another instance" in str(e):
            print(f"⚠️ Attempt {attempt + 1}/{max_retries}: Qdrant is locked by another instance")
            
            if attempt < max_retries - 1:
                # Check if lock file exists and try to remove it
                if os.path.exists(lock_file_path):
                    print(f"   Found lock file at {lock_file_path}")
                    print(f"   Attempting to remove stale lock file...")
                    try:
                        os.remove(lock_file_path)
                        print(f"   ✅ Lock file removed, retrying...")
                        time.sleep(0.5)  # Brief pause before retry
                    except Exception as lock_error:
                        print(f"   ⚠️ Could not remove lock file: {lock_error}")
                        print(f"   💡 Solution: Close other notebooks/kernels using Qdrant, or run:")
                        print(f"      rm -f {lock_file_path}")
                else:
                    print(f"   ⚠️ Lock file not found, but Qdrant reports it's locked")
                    print(f"   💡 Solution: Close other notebooks/kernels using Qdrant")
            else:
                print(f"\n❌ Failed to initialize Qdrant after {max_retries} attempts")
                print(f"\n💡 Solutions:")
                print(f"   1. Close other notebooks/kernels that might be using Qdrant")
                print(f"   2. Run this command in terminal: rm -f {lock_file_path}")
                print(f"   3. Restart your Jupyter kernel")
                raise
        else:
            # Some other RuntimeError, re-raise it
            raise

if qdrant is None:
    raise RuntimeError("Failed to initialize Qdrant client")

# Recreate knowledge_space collection
print("\n📦 Setting up collections...")
try:
    qdrant.delete_collection(collection_name="knowledge_space")
    print("   ✅ Deleted existing 'knowledge_space' collection")
except Exception as e:
    print(f"   ℹ️ No existing 'knowledge_space' collection to delete: {e}")

try:
    qdrant.create_collection(
        collection_name="knowledge_space",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    print("✅ Qdrant collection 'knowledge_space' created")
except Exception as e:
    print(f"⚠️ Error creating collection: {e}")
    # Check if collection already exists
    try:
        collections = qdrant.get_collections()
        if any(c.name == "knowledge_space" for c in collections.collections):
            print("   ℹ️ Collection 'knowledge_space' already exists, using existing collection")
        else:
            raise
    except:
        raise


⚠️ Attempt 1/3: Qdrant is locked by another instance
   Found lock file at qdrant_data/.lock
   Attempting to remove stale lock file...
   ✅ Lock file removed, retrying...
✅ Qdrant client initialized successfully

📦 Setting up collections...
   ✅ Deleted existing 'knowledge_space' collection
✅ Qdrant collection 'knowledge_space' created


In [13]:
# Index FAQ data (keep as-is)
faq_texts = [
    f"FAQ | Question: {row['question']} | Answer: {row['answer']}"
    for _, row in faq.iterrows()
]
faq_vectors = embedder.encode(faq_texts, show_progress_bar=False)
faq_payloads = []

for idx, row in faq.iterrows():
    faq_payloads.append({
        "type": "faq",
        "id_kind": "faq",
        "faq_id": int(idx),
        "question": row["question"], 
        "answer": row["answer"],
    })

print(f"✅ Encoded {len(faq_vectors)} FAQ items")


✅ Encoded 22 FAQ items


In [14]:
# Index graph nodes (for reference, but we'll use NetworkX for graph search)
# We still index them for potential hybrid search, but primary search is via NetworkX

graph_nodes = list(G.nodes(data=True))
graph_texts = []
graph_payloads = []

for i, (name, attrs) in enumerate(graph_nodes):
    ntype = attrs.get("type", "unknown")
    neighbors = list(G.successors(name)) + list(G.predecessors(name))
    neighbors_str = ", ".join(neighbors) if neighbors else "None"
    
    text = f"GRAPH_NODE | Type: {ntype} | Name: {name} | Neighbors: {neighbors_str}"
    graph_texts.append(text)
    
    graph_payloads.append({
        "type": "graph_node",
        "id_kind": "graph_node",
        "node_name": name,
        "node_type": ntype,
        "neighbors": neighbors,
    })

graph_vectors = embedder.encode(graph_texts, show_progress_bar=False)
print(f"✅ Encoded {len(graph_vectors)} graph nodes")


✅ Encoded 23 graph nodes


In [15]:
# Upsert FAQ and graph nodes to Qdrant
points = []

# FAQ points
for i, (vec, payload) in enumerate(zip(faq_vectors, faq_payloads)):
    points.append(
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload=payload
        )
    )

offset = len(points)

# Graph-node points
for j, (vec, payload) in enumerate(zip(graph_vectors, graph_payloads)):
    points.append(
        PointStruct(
            id=offset + j,
            vector=vec.tolist(),
            payload=payload
        )
    )

qdrant.upsert(collection_name="knowledge_space", points=points)
print(f"✅ Upserted {len(points)} total points into 'knowledge_space'")
print(f"   FAQ: {len(faq_vectors)} points")
print(f"   Graph nodes: {len(graph_vectors)} points")


✅ Upserted 45 total points into 'knowledge_space'
   FAQ: 22 points
   Graph nodes: 23 points


## Prompt Building Function
Build prompt WITHOUT style (removed style section)


In [16]:
# Build prompt WITHOUT style (removed style section)
def build_prompt_enhanced(
    email_text: str,
    intents: List[str],
    artifacts: List[str],
    faq_hits: List[Dict],
    graph_replies: List[str]  # Replies from graph RAG search
):
    """
    Build prompt with content context (FAQs, graph replies).
    NO style section (removed).
    
    Args:
        email_text: Incoming email
        intents: Detected intents
        artifacts: Detected artifacts
        faq_hits: FAQ search results
        graph_replies: Replies from graph RAG intersection search
    """
    faq_section = "\n".join([
        f"{i+1}. Q: {f['question']}\n   A: {f['answer']}"
        for i, f in enumerate(faq_hits)
    ]) or "None"
    
    # Graph replies section (NEW: replies from intersection search)
    graph_replies_section = ""
    if graph_replies:
        graph_replies_section = "\n".join([
            f"{i+1}. Similar Reply Example {i+1}:\n   \"{reply[:300]}...\""
            for i, reply in enumerate(graph_replies[:5])  # Top 5 replies
        ])
    else:
        graph_replies_section = "None"
    
    intents_str = ", ".join(intents)
    artifacts_str = ", ".join(artifacts) if artifacts else "None"
    
    prompt = f"""
You are **Zubair**, a graduate student known for being polite, proactive, and clear in communication.

Your job is to draft a short, natural, and professional email reply.

Keep it warm but not overly formal — think of how a thoughtful student would respond to a professor, coordinator, or peer.

---

✉️ **Incoming Email**
\"\"\"{email_text}\"\"\"

🎯 **Detected Intents**: {intents_str}
📎 **Relevant Artifacts**: {artifacts_str}

📘 **Relevant FAQs** (Content Context)
{faq_section}

📧 **Similar Email Replies** (Graph RAG Context - examples of how similar emails were replied to)
{graph_replies_section}

---

Write your reply:
- Use similar tone and structure as the example replies above
- Acknowledge the sender and context
- If an action is requested, confirm or ask a polite follow-up question
- Include relevant artifacts/links if needed (based on detected artifacts)
- Keep the reply under 120 words
- Do NOT invent facts — only use what's in context
- End with: "Best Regards,\\nZubair"
"""
    return prompt

print("✅ Enhanced prompt builder loaded!")


✅ Enhanced prompt builder loaded!


## Main Answer Email Function
Complete pipeline with new methodology


In [17]:
# MAIN ANSWER EMAIL FUNCTION (Enhanced)
def answer_email_enhanced(
    email_text: str,
    top_k: int = 6,
    show_context: bool = True
):
    """
    Enhanced email answering with new methodology:
    1. Enhanced intent + artifact classification (BOTH intents AND artifacts)
    2. FAQ search (Qdrant) - ALWAYS retrieves FAQs
    3. Graph RAG search (NetworkX) - intersection of intents and artifacts
    4. Extract replies from matching emails
    5. Build prompt WITHOUT style
    6. Generate reply
    """
    
    print("\n" + "="*80)
    print("🚀 ENHANCED EMAIL ANSWERING PIPELINE")
    print("="*80 + "\n")
    
    # Step 1: Enhanced intent + artifact classification
    print("📋 Step 1: Intent + Artifact Classification")
    print("-"*80)
    classification_result = classify_intent_and_artifacts(
        email_text=email_text,
        available_intents=list(unique_intents),
        available_artifacts=list(unique_artifacts),
        artifact_dict=ARTIFACT_DICTIONARY
    )
    
    intents = classification_result["intents"]
    artifacts = classification_result["artifacts"]
    primary_intent = intents[0] if intents else "general_inquiry"
    
    if show_context:
        print(f"   ✅ Intents: {intents}")
        print(f"   ✅ Artifacts: {artifacts}")
        print(f"   ✅ Primary Intent: {primary_intent}\n")
    
    # Step 2: Embed query for vector search
    q_vec = embedder.encode([email_text])[0].tolist()
    
    # Step 3: Search Qdrant for FAQs ONLY
    print("📋 Step 2: FAQ Search")
    print("-"*80)
    try:
        from qdrant_client.models import Filter, FieldCondition, MatchValue
        faq_search_results = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,
            limit=top_k,
            query_filter=Filter(
                must=[FieldCondition(key="type", match=MatchValue(value="faq"))]
            )
        ).points
    except (AttributeError, TypeError, ImportError):
        # Fallback: search all and filter manually
        all_hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k * 2,
            with_payload=True
        )
        faq_search_results = [h for h in all_hits if h.payload.get("type") == "faq"][:top_k]
    
    # Extract FAQ hits
    faq_hits = []
    for h in faq_search_results:
        p = h.payload
        if p.get("type") == "faq":
            faq_hits.append({"score": h.score, **p})
    
    if show_context:
        print(f"   ✅ FAQ hits: {len(faq_hits)}\n")
    
    # Step 4: Graph RAG search (NEW: intersection of intents and artifacts)
    print("📋 Step 3: Graph RAG Search (Intent + Artifact Intersection)")
    print("-"*80)
    
    matching_email_ids = []
    graph_replies = []
    
    if intents and artifacts:
        matching_email_ids = find_emails_by_intent_artifact_intersection(
            intents=intents,
            artifacts=artifacts,
            node_to_email_ids=node_to_email_ids
        )
        
        # Extract replies from matching emails
        for email_id in matching_email_ids:
            reply = email_id_to_reply.get(email_id)
            if reply:
                graph_replies.append(reply)
        
        if show_context:
            print(f"   ✅ Matching emails: {len(matching_email_ids)}")
            print(f"   ✅ Replies extracted: {len(graph_replies)}")
            if graph_replies:
                print(f"\n   📧 Sample replies:")
                for i, reply in enumerate(graph_replies[:3], 1):
                    print(f"      {i}. {reply[:100]}...")
    else:
        if show_context:
            print("   ⚠️ Need at least 1 intent AND 1 artifact for graph search")
    
    print()
    
    # Step 5: Build enhanced prompt (NO style)
    print("📋 Step 4: Building Prompt")
    print("-"*80)
    prompt = build_prompt_enhanced(
        email_text=email_text,
        intents=intents,
        artifacts=artifacts,
        faq_hits=faq_hits,
        graph_replies=graph_replies
    )
    
    if show_context:
        print("   ✅ Prompt built (without style section)\n")
    
    # Step 6: Generate reply
    print("📋 Step 5: Generating Reply")
    print("-"*80)
    resp = ollama.chat(
        model="qwen2.5:14b-instruct-q4_K_M",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]
    
    if show_context:
        print("   ✅ Reply generated\n")
    
    # Step 7: Confidence scoring
    top_score = faq_hits[0]["score"] if faq_hits else 0.0
    auto_send = top_score > 0.85
    
    return {
        "intents": intents,
        "artifacts": artifacts,
        "primary_intent": primary_intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_emails_found": len(matching_email_ids),
        "graph_replies_used": len(graph_replies),
        "matching_email_ids": matching_email_ids[:10],  # First 10 for debugging
    }

print("✅ Enhanced answer_email function loaded!")


✅ Enhanced answer_email function loaded!


In [18]:
# TEST 1: ML/Analytics Resume Request
print("\n" + "="*80)
print("TEST 1: ML/Analytics Resume Request")
print("="*80)

test_email_1 = """Hi Zubair,
Thank you for reaching out. To proceed with your interest in the Advanced Analytics position at Google, kindly share your resume and provide the right time to connect with you for an online meeting.
Regards,
Arjun Das
Talent Acquisition
Google Inc."""

result_1 = answer_email_enhanced(test_email_1, show_context=True)

print("\n" + "="*80)
print("📧 FINAL RESULT")
print("="*80)
print(f"Intents: {result_1['intents']}")
print(f"Artifacts: {result_1['artifacts']}")
print(f"Confidence: {result_1['top_score']:.3f}")
print(f"Auto-send: {result_1['auto_send']}")
print(f"Graph emails found: {result_1['graph_emails_found']}")
print(f"\n✍️ Generated Reply:")
print("-"*80)
print(result_1['reply'])
print("-"*80)



TEST 1: ML/Analytics Resume Request

🚀 ENHANCED EMAIL ANSWERING PIPELINE

📋 Step 1: Intent + Artifact Classification
--------------------------------------------------------------------------------
🔍 LLM raw response: {
    "intents": ["send_materials", "schedule"],
    "artifacts": ["ds_resume", "calendly"]
}...
✅ Classified intents: ['send_materials', 'schedule']
✅ Classified artifacts: ['ds_resume', 'calendly']
   ✅ Intents: ['send_materials', 'schedule']
   ✅ Artifacts: ['ds_resume', 'calendly']
   ✅ Primary Intent: send_materials

📋 Step 2: FAQ Search
--------------------------------------------------------------------------------
   ✅ FAQ hits: 6

📋 Step 3: Graph RAG Search (Intent + Artifact Intersection)
--------------------------------------------------------------------------------

🔍 Searching for emails with:
   Intents: ['send_materials', 'schedule']
   Artifacts: ['ds_resume', 'calendly']
   Intent 'send_materials': 25 emails
   Intent 'schedule': 20 emails
   Artifact '

In [19]:
# TEST 2: LinkedIn Profile Request
print("\n" + "="*80)
print("TEST 2: LinkedIn Profile Request")
print("="*80)

test_email_2 = """Hi Zubair, I hope you're doing well. Can I get your linkedin profile to connect?"""

result_2 = answer_email_enhanced(test_email_2, show_context=True)

print("\n" + "="*80)
print("📧 FINAL RESULT")
print("="*80)
print(f"Intents: {result_2['intents']}")
print(f"Artifacts: {result_2['artifacts']}")
print(f"Confidence: {result_2['top_score']:.3f}")
print(f"Auto-send: {result_2['auto_send']}")
print(f"Graph emails found: {result_2['graph_emails_found']}")
print(f"\n✍️ Generated Reply:")
print("-"*80)
print(result_2['reply'])
print("-"*80)



TEST 2: LinkedIn Profile Request

🚀 ENHANCED EMAIL ANSWERING PIPELINE

📋 Step 1: Intent + Artifact Classification
--------------------------------------------------------------------------------
🔍 LLM raw response: {
    "intents": ["request_info"],
    "artifacts": ["linkedin_profile"]
}...
✅ Classified intents: ['request_info']
✅ Classified artifacts: ['linkedin_profile']
   ✅ Intents: ['request_info']
   ✅ Artifacts: ['linkedin_profile']
   ✅ Primary Intent: request_info

📋 Step 2: FAQ Search
--------------------------------------------------------------------------------
   ✅ FAQ hits: 6

📋 Step 3: Graph RAG Search (Intent + Artifact Intersection)
--------------------------------------------------------------------------------

🔍 Searching for emails with:
   Intents: ['request_info']
   Artifacts: ['linkedin_profile']
   Intent 'request_info': 15 emails
   Artifact 'linkedin_profile': 10 emails

   ✅ Found 5 emails in intersection

   🔍 Trying specific intent+artifact combination

In [20]:
# TEST 3: Meeting Reschedule Request
print("\n" + "="*80)
print("TEST 3: Meeting Reschedule Request")
print("="*80)

test_email_3 = """Hi Zubair,
Hope you're doing well. I wanted to check if our meeting scheduled for tomorrow is still on.
If not, could you suggest another time that works for you?
Thanks,
Alex"""

result_3 = answer_email_enhanced(test_email_3, show_context=True)

print("\n" + "="*80)
print("📧 FINAL RESULT")
print("="*80)
print(f"Intents: {result_3['intents']}")
print(f"Artifacts: {result_3['artifacts']}")
print(f"Confidence: {result_3['top_score']:.3f}")
print(f"Auto-send: {result_3['auto_send']}")
print(f"Graph emails found: {result_3['graph_emails_found']}")
print(f"\n✍️ Generated Reply:")
print("-"*80)
print(result_3['reply'])
print("-"*80)



TEST 3: Meeting Reschedule Request

🚀 ENHANCED EMAIL ANSWERING PIPELINE

📋 Step 1: Intent + Artifact Classification
--------------------------------------------------------------------------------
🔍 LLM raw response: {
    "intents": ["reschedule", "confirm"],
    "artifacts": ["calendly"]
}...
✅ Classified intents: ['reschedule', 'confirm']
✅ Classified artifacts: ['calendly']
   ✅ Intents: ['reschedule', 'confirm']
   ✅ Artifacts: ['calendly']
   ✅ Primary Intent: reschedule

📋 Step 2: FAQ Search
--------------------------------------------------------------------------------
   ✅ FAQ hits: 6

📋 Step 3: Graph RAG Search (Intent + Artifact Intersection)
--------------------------------------------------------------------------------

🔍 Searching for emails with:
   Intents: ['reschedule', 'confirm']
   Artifacts: ['calendly']
   Intent 'reschedule': 15 emails
   Intent 'confirm': 9 emails
   Artifact 'calendly': 15 emails

   ✅ Found 0 emails in intersection
   ✅ Matching emails: 0
 

In [21]:
# TEST 4: Software Engineering Resume Request
print("\n" + "="*80)
print("TEST 4: Software Engineering Resume Request")
print("="*80)

test_email_4 = """Hello Zubair,
We're currently on the lookout for skilled individuals in software engineering, and your profile caught our eye. Would you mind sharing your resume with us? We'd love to explore potential opportunities together.
Best,
Salman Khan
Bhai Dynamics"""

result_4 = answer_email_enhanced(test_email_4, show_context=True)

print("\n" + "="*80)
print("📧 FINAL RESULT")
print("="*80)
print(f"Intents: {result_4['intents']}")
print(f"Artifacts: {result_4['artifacts']}")
print(f"Confidence: {result_4['top_score']:.3f}")
print(f"Auto-send: {result_4['auto_send']}")
print(f"Graph emails found: {result_4['graph_emails_found']}")
print(f"\n✍️ Generated Reply:")
print("-"*80)
print(result_4['reply'])
print("-"*80)



TEST 4: Software Engineering Resume Request

🚀 ENHANCED EMAIL ANSWERING PIPELINE

📋 Step 1: Intent + Artifact Classification
--------------------------------------------------------------------------------
🔍 LLM raw response: {
    "intents": ["send_materials"],
    "artifacts": ["swe_resume"]
}...
✅ Classified intents: ['send_materials']
✅ Classified artifacts: ['swe_resume']
   ✅ Intents: ['send_materials']
   ✅ Artifacts: ['swe_resume']
   ✅ Primary Intent: send_materials

📋 Step 2: FAQ Search
--------------------------------------------------------------------------------
   ✅ FAQ hits: 6

📋 Step 3: Graph RAG Search (Intent + Artifact Intersection)
--------------------------------------------------------------------------------

🔍 Searching for emails with:
   Intents: ['send_materials']
   Artifacts: ['swe_resume']
   Intent 'send_materials': 25 emails
   Artifact 'swe_resume': 10 emails

   ✅ Found 10 emails in intersection

   🔍 Trying specific intent+artifact combinations...
 

In [22]:
# TEST 5: Multiple Artifacts Request (GitHub + Portfolio)
print("\n" + "="*80)
print("TEST 5: Multiple Artifacts Request")
print("="*80)

test_email_5 = """Hi Zubair,
I'm interested in seeing your work samples. Can you share your GitHub and portfolio links?
Thanks,
Sarah"""

result_5 = answer_email_enhanced(test_email_5, show_context=True)

print("\n" + "="*80)
print("📧 FINAL RESULT")
print("="*80)
print(f"Intents: {result_5['intents']}")
print(f"Artifacts: {result_5['artifacts']}")
print(f"Confidence: {result_5['top_score']:.3f}")
print(f"Auto-send: {result_5['auto_send']}")
print(f"Graph emails found: {result_5['graph_emails_found']}")
print(f"\n✍️ Generated Reply:")
print("-"*80)
print(result_5['reply'])
print("-"*80)



TEST 5: Multiple Artifacts Request

🚀 ENHANCED EMAIL ANSWERING PIPELINE

📋 Step 1: Intent + Artifact Classification
--------------------------------------------------------------------------------
🔍 LLM raw response: {
    "intents": ["send_materials"],
    "artifacts": ["github", "portfolio"]
}...
✅ Classified intents: ['send_materials']
✅ Classified artifacts: ['github', 'portfolio']
   ✅ Intents: ['send_materials']
   ✅ Artifacts: ['github', 'portfolio']
   ✅ Primary Intent: send_materials

📋 Step 2: FAQ Search
--------------------------------------------------------------------------------
   ✅ FAQ hits: 6

📋 Step 3: Graph RAG Search (Intent + Artifact Intersection)
--------------------------------------------------------------------------------

🔍 Searching for emails with:
   Intents: ['send_materials']
   Artifacts: ['github', 'portfolio']
   Intent 'send_materials': 25 emails
   Artifact 'github': 10 emails
   Artifact 'portfolio': 5 emails

   ✅ Found 5 emails in intersection

In [2]:
# """
# Hello Zubair,

# We're currently on the lookout for skilled individuals in data science, and your profile caught our eye. Would you mind sharing your resume with us? We'd love to explore potential opportunities together.

# Also, can you send your calendlly link?

# Best,
# Salman Khan
# Bhai Dynamics
# """
# TEST 5: Multiple Artifacts Request (GitHub + Portfolio)


test_email_5 = """
Hello Zubair,

We're currently on the lookout for skilled individuals in data science, and your profile caught our eye. Would you mind sharing your resume with us? We'd love to explore potential opportunities together.

Also, can you send your calendlly link?

Best,
Salman Khan
Bhai Dynamics
"""

result_5 = answer_email_enhanced(test_email_5, show_context=True)

print("\n" + "="*80)
print("📧 FINAL RESULT")
print("="*80)
print(f"Intents: {result_5['intents']}")
print(f"Artifacts: {result_5['artifacts']}")
print(f"Confidence: {result_5['top_score']:.3f}")
print(f"Auto-send: {result_5['auto_send']}")
print(f"Graph emails found: {result_5['graph_emails_found']}")
print(f"\n✍️ Generated Reply:")
print("-"*80)
print(result_5['reply'])
print("-"*80)


NameError: name 'answer_email_enhanced' is not defined

In [ ]:
""